In [ ]:
# ============================================================
# Add masked reconstruction plot (masked_reconstruction_mse)
# - Builds a long-form DataFrame for a single PID from the loader batch
# - Calls your masked_reconstruction_mse(..., pid_target=PID) to plot
# - Integrated into run_split1_pipeline via flags
# ============================================================

import os, glob, json
import torch
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil, sqrt
from argparse import Namespace
# ---- project modules ----
from surprise_data.feather_domains import build_feather_split_loaders, build_loaders_for_vis
from surprise_configs_feather import build_feather_cfg_from_args
from surprise_models.build import build_model
from surprise_lit.downstream import STRaTSLit

# ---- PaCMAP ----
try:
    import pacmap
except ImportError as e:
    raise ImportError("Run `!pip install pacmap` in a notebook cell, then re-run.") from e


# ============================================================
# Pretty names / format helpers
# ============================================================
PRETTY_VAR_NAMES = {
    "gcs": "GCS",
    "heart_rate": "Heart Rate",
    "map": "MAP",
    "resp_rate": "Respiratory Rate",
    "temperature": "Temperature",
    "weight": "Weight",
    "albumin": "Albumin",
    "bilirubin": "Bilirubin",
    "creatinine": "Creatinine",
    "fio2": "FiO2",
    "glucose": "Glucose",
    "hematocrit": "Hematocrit",
    "lactate": "Lactate",
    "pao2": "PaO2",
    "ph": "pH",
    "platelets": "Platelets",
    "sodium": "Sodium",
    "urine": "Urine",
    "wbc": "WBC",
    "a10": "A10",
    "a_drug": "A Drug",
    "a_supplements": "A Supplements",
    "b": "B",
    "c01": "C01",
    "c01_etc": "C01 Etc",
    "c_else": "C Else",
    "h": "H",
    "l": "L",
    "m": "M",
    "n": "N",
    "r": "R",
    "v": "V",
    "antibiotics": "Antibiotics",
    "fluid": "Fluid",
    "ventilator": "Ventilator",
}

def format_model_name(model_name: str) -> str:
    mapping = {
        "strats": "STraTS",
        "surprise": "Surprise",
        "surprise_vt": "SurpVT",
        "surprise_vttg": "SurpVTTG",
    }
    return mapping.get(model_name, model_name)

def format_source_name(source_domain: str) -> str:
    return "MIMIC" if source_domain.lower() == "mimic" else "eICU"

def invert_var_names(var_names: dict[int, str]) -> dict[str, int]:
    return {v: k for k, v in var_names.items()}

def pretty_var_label_from_id(var_id: int, var_names: dict[int, str] | None = None) -> str:
    raw = var_names.get(var_id, f"var {var_id}") if var_names else f"var {var_id}"
    return PRETTY_VAR_NAMES.get(raw, raw.replace("_", " ").title())

# ============================================================
# Fixed y-ranges for selected variables
# ============================================================
FIXED_VAR_YLIMS = {
    "heart_rate": (30.0, 160.0),
    "map": (35.0, 165.0),
    "glucose": (70.0, 250.0),
    "lactate": (0.6, 3.5),
}


def get_fixed_ylim_for_var(var_id, var_names=None, fixed_var_ylims=None):
    if fixed_var_ylims is None:
        fixed_var_ylims = FIXED_VAR_YLIMS
    raw = var_names.get(var_id, f"var_{var_id}") if var_names is not None else f"var_{var_id}"
    return fixed_var_ylims.get(raw, None)

# ============================================================
# Optional inverse transform helpers for plotting
# NOTE:
# - value inverse depends on how you normalized each variable.
# - time inverse assumes normalized time was in [0,1] from original [0,2880].
# ============================================================
def inverse_time_default(t_norm):
    # normalized [0,1] -> original minutes [0,2880]
    return np.asarray(t_norm) * 2880.0

def inverse_values_per_var(values, var_ids, inverse_value_map=None):
    """
    values: np.ndarray shape [N]
    var_ids: np.ndarray shape [N]
    inverse_value_map:
        dict[int, callable]
        each callable receives vector of normalized values and returns original-scale values
    """
    values = np.asarray(values).copy()
    var_ids = np.asarray(var_ids)

    if inverse_value_map is None:
        return values

    out = values.astype(float).copy()
    for vid, fn in inverse_value_map.items():
        m = (var_ids == vid)
        if np.any(m):
            out[m] = fn(out[m])
    return out

def make_inverse_value_map_from_stats(stats):
    per_item = stats["value"]["per_item"]
    g_mean, g_std = stats["value"]["global"]
    cat_set = set(stats["value"].get("categorical", []))

    if g_std < 1e-12:
        g_std = 1.0

    inverse_value_map = {}

    all_itemids = set(per_item.keys()) | set(cat_set)

    for itemid in all_itemids:
        if itemid in cat_set:
            inverse_value_map[itemid] = lambda x: x
        else:
            m, s = per_item.get(itemid, (g_mean, g_std))
            if s < 1e-12:
                s = 1.0
            inverse_value_map[itemid] = (lambda x, mu=m, sd=s: x * sd + mu)

    return inverse_value_map

def inverse_static_vector(st, stats):
    st = np.asarray(st).copy().astype(np.float32)

    age_m, age_s = stats["static"]["age"]
    height_m, height_s = stats["static"]["height"]

    if age_s < 1e-12:
        age_s = 1.0
    if height_s < 1e-12:
        height_s = 1.0

    st[0] = st[0] * age_s + age_m
    st[2] = st[2] * height_s + height_m
    return st

# ============================================================
# UPDATED: build per-PID long DataFrame from tensors + mask
# with optional inverse transforms for plotting scale
# ============================================================
def pid_tensors_to_long_df(
    *,
    pid: int,
    times1: torch.Tensor,     # [1,L]
    varis1: torch.Tensor,     # [1,L]
    values1: torch.Tensor,    # [1,L]
    pad1: torch.Tensor,       # [1,L] bool True=pad
    redundant_mask1: torch.Tensor,  # [1,L] bool True=redundant
    inverse_time_fn=None,
    inverse_value_map=None,
) -> pd.DataFrame:
    """
    Returns df with columns: pid, times, varis, values, mask
    - drops padded positions
    - mask column corresponds to redundant_mask (True = masked)
    - times/values can be inverse-transformed for plotting
    """
    t = times1.detach().cpu().view(-1).numpy()
    f = varis1.detach().cpu().view(-1).numpy()
    v = values1.detach().cpu().view(-1).numpy()
    pad = pad1.detach().cpu().view(-1).numpy().astype(bool)
    m = redundant_mask1.detach().cpu().view(-1).numpy().astype(bool)

    keep = ~pad
    t = t[keep]
    f = f[keep]
    v = v[keep]
    m = m[keep]

    if inverse_time_fn is not None:
        t = inverse_time_fn(t)

    v = inverse_values_per_var(v, f, inverse_value_map=inverse_value_map)

    df = pd.DataFrame({
        "pid": pid,
        "times": t,
        "varis": f,
        "values": v,
        "mask": m,
    })
    return df


# ============================================================
# UPDATED: masked_reconstruction_mse
# - pretty titles
# - selected vars only
# - original time/value scale
# - no pid in suptitle
# ============================================================
def masked_reconstruction_plot_seaborn(
    df_pid,
    var_names=None,
    plot_only_vars=None,
    time_col='times',
    var_col='varis',
    value_col='values',
    mask_col='mask',
    suptitle=None,
    save_path=None,
):
    sns.set(style="whitegrid", context="talk")

    uniq_vars = df_pid[var_col].unique()
    if plot_only_vars is not None:
        uniq_vars = [v for v in uniq_vars if v in set(plot_only_vars)]

    nplots = len(uniq_vars)
    cols = int(ceil(sqrt(nplots)))
    rows_ = int(ceil(nplots / cols))

    fig, axes = plt.subplots(
        rows_, cols,
        figsize=(5 * cols, 3.5 * rows_),
    )

    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.reshape(rows_, cols)

    for idx, var_id in enumerate(uniq_vars):
        r, c = divmod(idx, cols)
        ax = axes[r, c]

        g = df_pid[df_pid[var_col] == var_id].copy()
        g = g.sort_values(time_col)

        g["type"] = np.where(g[mask_col], "Masked", "Unmasked")

        # background (all)
        sns.scatterplot(
            data=g,
            x=time_col,
            y=value_col,
            color="lightgray",
            alpha=0.3,
            s=10,
            ax=ax,
            legend=False,
        )

        # masked
        sns.scatterplot(
            data=g[g["type"] == "Masked"],
            x=time_col,
            y=value_col,
            color="red",
            marker="X",
            s=40,
            ax=ax,
            label="Masked",
        )

        # valid (위)
        sns.scatterplot(
            data=g[g["type"] == "Unmasked"],
            x=time_col,
            y=value_col,
            color="blue",
            s=40,
            ax=ax,
            label="Unmasked",
        )

        # reconstruction
        g_un = g[g["type"] == "Unmasked"]
        if len(g_un) >= 2:
            t_k = g_un[time_col].to_numpy()
            y_k = g_un[value_col].to_numpy()

            order = np.argsort(t_k)
            t_k = t_k[order]
            y_k = y_k[order]

            t_all = g[time_col].to_numpy()
            y_hat = np.interp(t_all, t_k, y_k)

            sns.lineplot(
                x=t_all,
                y=y_hat,
                color="black",
                linewidth=2,
                ax=ax,
                label="Reconstruction",
            )

        ax.set_title(pretty_var_label_from_id(var_id, var_names))
        ax.set_xlim(0, 2880)

    # remove empty axes
    for j in range(nplots, rows_ * cols):
        r, c = divmod(j, cols)
        axes[r, c].axis("off")

    # global legend
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=3,
        frameon=False,
    )

    if suptitle:
        fig.suptitle(suptitle, y=0.98)

    fig.tight_layout(rect=[0, 0.05, 1, 0.95])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

# ============================================================
# Shared reconstruction computation for one PID dataframe
# ============================================================
def compute_reconstruction_dict(
    df_pid,
    *,
    time_col="times",
    var_col="varis",
    value_col="values",
    mask_col="mask",
):
    """
    Compute reconstruction inputs exactly once, so single/multi plots
    use the same logic.

    Returns
    -------
    dict[var_id] -> {
        "t_all", "y_all",
        "t_un", "y_un",
        "t_mk", "y_mk",
        "y_hat_all",
    }
    """
    out = {}

    for var_id, g in df_pid.groupby(var_col, sort=False):
        g = g.sort_values(time_col)

        t_all = g[time_col].to_numpy()
        y_all = g[value_col].to_numpy()

        g_un = g[~g[mask_col]]
        g_mk = g[g[mask_col]]

        t_un = g_un[time_col].to_numpy()
        y_un = g_un[value_col].to_numpy()

        t_mk = g_mk[time_col].to_numpy()
        y_mk = g_mk[value_col].to_numpy()

        if len(g_un) >= 1:
            if len(g_un) == 1:
                y_hat_all = np.full_like(y_all, fill_value=y_un[0], dtype=float)
            else:
                t_k = g_un[time_col].to_numpy()
                y_k = g_un[value_col].to_numpy()

                if len(np.unique(t_k)) != len(t_k):
                    tmp = (
                        pd.DataFrame({time_col: t_k, "yk": y_k})
                        .groupby(time_col)["yk"].mean()
                        .reset_index()
                    )
                    t_k = tmp[time_col].to_numpy()
                    y_k = tmp["yk"].to_numpy()

                order = np.argsort(t_k)
                t_k = t_k[order]
                y_k = y_k[order]

                y_hat_all = np.interp(t_all, t_k, y_k)
        else:
            y_hat_all = None

        out[var_id] = {
            "t_all": t_all,
            "y_all": y_all,
            "t_un": t_un,
            "y_un": y_un,
            "t_mk": t_mk,
            "y_mk": y_mk,
            "y_hat_all": y_hat_all,
        }

    return out

# ============================================================
# UPDATED single-variant mask plot with fixed y-ranges
# ============================================================
def masked_reconstruction_mse(
    df,
    var_names=None,
    pid_filter=None,
    pid_target=None,
    plot_only_vars=None,
    time_col='times',
    var_col='varis',
    value_col='values',
    mask_col='mask',
    time_unit="min",
    figsize_per_plot=(4.5, 3.2),
    sharey=False,
    save_path=None,
    suptitle=None,
    fixed_var_ylims=None,
):
    required = {'pid', time_col, var_col, value_col, mask_col}
    if not required.issubset(df.columns):
        raise ValueError(f"df must contain columns {required}")

    d = df.copy()

    if pid_filter is not None:
        if np.isscalar(pid_filter):
            d = d[d['pid'] == pid_filter]
        else:
            d = d[d['pid'].isin(pid_filter)]

    if d.empty:
        raise ValueError("Filtered df is empty.")

    d = d.sort_values(['pid', var_col, time_col]).reset_index(drop=True)

    # ---------- masked-only MSE summary ----------
    rows = []
    sse_total = 0.0
    n_masked_total = 0
    n_pid_var_valid = 0

    for (pid, varid), g in d.groupby(['pid', var_col], sort=False):
        g = g.sort_values(time_col)

        g_un = g[~g[mask_col]]
        g_mk = g[g[mask_col]]

        n_all = len(g)
        n_un = len(g_un)
        n_mk = len(g_mk)

        if n_mk == 0 or n_un == 0:
            mse_mk = np.nan
            rows.append({
                'pid': pid, var_col: varid,
                'mse_masked': mse_mk,
                'n_masked': n_mk, 'n_unmasked': n_un, 'n_all': n_all
            })
            continue

        if n_un == 1:
            y_hat_mk = np.full(shape=n_mk, fill_value=g_un[value_col].iloc[0], dtype=float)
        else:
            t_k = g_un[time_col].to_numpy()
            y_k = g_un[value_col].to_numpy()

            if len(np.unique(t_k)) != len(t_k):
                tmp = (
                    pd.DataFrame({time_col: t_k, 'yk': y_k})
                    .groupby(time_col)['yk'].mean()
                    .reset_index()
                )
                t_k = tmp[time_col].to_numpy()
                y_k = tmp['yk'].to_numpy()

            order = np.argsort(t_k)
            t_k = t_k[order]
            y_k = y_k[order]

            t_mk = g_mk[time_col].to_numpy()
            y_hat_mk = np.interp(t_mk, t_k, y_k)

        y_mk = g_mk[value_col].to_numpy()
        se = (y_mk - y_hat_mk) ** 2
        mse_mk = float(np.mean(se))

        sse_total += float(np.sum(se))
        n_masked_total += int(n_mk)
        n_pid_var_valid += 1

        rows.append({
            'pid': pid, var_col: varid,
            'mse_masked': mse_mk,
            'n_masked': n_mk, 'n_unmasked': n_un, 'n_all': n_all
        })

    pid_var_detail = pd.DataFrame(rows)

    agg = (
        pid_var_detail
        .groupby(var_col)
        .agg(
            n_pid_valid=('mse_masked', lambda s: int(s.notna().sum())),
            n_masked_points=('n_masked', 'sum'),
            mse_masked_mean=('mse_masked', 'mean'),
            mse_masked_std=('mse_masked', 'std'),
        )
        .reset_index()
    )

    agg['var_name'] = agg[var_col].map(
        lambda x: pretty_var_label_from_id(x, var_names=var_names)
    )

    var_mse_df = agg[
        [var_col, 'var_name', 'n_masked_points', 'n_pid_valid', 'mse_masked_mean', 'mse_masked_std']
    ].sort_values('mse_masked_mean', ascending=False).reset_index(drop=True)

    overall = {
        'overall_mse_masked': (sse_total / n_masked_total) if n_masked_total > 0 else np.nan,
        'overall_n_masked_points': int(n_masked_total),
        'overall_n_pid_var_valid': int(n_pid_var_valid),
    }

    # ---------- plotting ----------
    if pid_target is not None:
        df_pid = d[d['pid'] == pid_target].copy()
        if df_pid.empty:
            raise ValueError(f"pid={pid_target} not found.")

        df_pid = df_pid.sort_values([var_col, time_col])
        recon_dict = compute_reconstruction_dict(
            df_pid,
            time_col=time_col,
            var_col=var_col,
            value_col=value_col,
            mask_col=mask_col,
        )

        uniq_vars = list(recon_dict.keys())
        if plot_only_vars is not None:
            plot_set = set(plot_only_vars)
            uniq_vars = [v for v in uniq_vars if v in plot_set]

        nplots = len(uniq_vars)
        if nplots == 0:
            raise ValueError("No variables to plot after filtering (plot_only_vars).")

        cols = int(ceil(sqrt(nplots)))
        rows_ = int(ceil(nplots / cols))

        fig, axes = plt.subplots(
            rows_, cols,
            figsize=(figsize_per_plot[0] * cols, figsize_per_plot[1] * rows_ + 0.8),
            sharey=sharey
        )
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        axes = axes.reshape(rows_, cols)

        handles, labels = [], []

        for idx, var_id in enumerate(uniq_vars):
            r, c = divmod(idx, cols)
            ax = axes[r, c]
            rd = recon_dict[var_id]

            if len(rd["t_mk"]) > 0:
                ax.scatter(
                    rd["t_mk"], rd["y_mk"],
                    marker='.',
                    alpha=0.8,
                    color="#6BA06B",
                    label='Masked',
                    zorder=1
                )

            if len(rd["t_un"]) > 0:
                ax.scatter(
                    rd["t_un"], rd["y_un"],
                    marker='o',
                    alpha=1.0,
                    color="#f34030",
                    label='Unmasked',
                    zorder=2
                )

            # ===== extend reconstruction to full time range =====
            if len(rd["t_un"]) >= 1:

                t_grid = np.linspace(0.0, 2880.0, 500)

                if len(rd["t_un"]) == 1:
                    y_hat_grid = np.full_like(
                        t_grid,
                        fill_value=rd["y_un"][0],
                        dtype=float
                    )
                else:
                    t_k = rd["t_un"]
                    y_k = rd["y_un"]

                    if len(np.unique(t_k)) != len(t_k):
                        tmp = (
                            pd.DataFrame({"t": t_k, "y": y_k})
                            .groupby("t")["y"].mean()
                            .reset_index()
                        )
                        t_k = tmp["t"].to_numpy()
                        y_k = tmp["y"].to_numpy()

                    order = np.argsort(t_k)
                    t_k = t_k[order]
                    y_k = y_k[order]

                    # 🔥 핵심: grid 기반 interpolation
                    y_hat_grid = np.interp(t_grid, t_k, y_k)

                ax.plot(
                    t_grid,
                    y_hat_grid,
                    linewidth=1.2,
                    color="#f34030",
                    label='Reconstruction',
                    zorder=3
                )

            ax.set_title(pretty_var_label_from_id(var_id, var_names=var_names), fontsize=22)
            ax.set_xlabel(f"Time ({time_unit})", fontsize=14)
            ax.set_ylabel("Value", fontsize=14)
            ax.set_xlim(0.0, 2880.0)
            ax.tick_params(axis='both', labelsize=12)

            ylim = get_fixed_ylim_for_var(var_id, var_names=var_names, fixed_var_ylims=fixed_var_ylims)
            if ylim is not None:
                ax.set_ylim(*ylim)

            ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels:
                    handles.append(h)
                    labels.append(l)

        for j in range(nplots, rows_ * cols):
            r, c = divmod(j, cols)
            axes[r, c].axis('off')

        if handles:
            fig.legend(
                handles, labels,
                loc='lower center',
                bbox_to_anchor=(0.5, 0.01),
                ncol=min(len(labels), 4),
                frameon=False,
                fontsize=14
            )

        if suptitle is not None:
            fig.suptitle(suptitle, y=0.98, fontsize=22)
            fig.tight_layout(rect=[0.02, 0.10, 0.98, 0.94])
        else:
            fig.tight_layout(rect=[0.02, 0.10, 0.98, 0.98])

        if save_path is not None:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            fig.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.show()
        plt.close(fig)

    return var_mse_df, overall, pid_var_detail

# ============================================================
# Multi-variant overlay plot
# reconstruction + unmasked share same color per variant
# legend order: variant / variant / variant / masked
# ============================================================
from matplotlib.lines import Line2D

VARIANT_COLOR_MAP = {
    "Surprise": "#f34030",
    "SurpVT": "#2D6CDF",
    "SurpVTTG": "#7A3FB0",
    "STraTS": "#E68613",
    "Transformer": "#1F9D8A",
    "Transmean": "#8A5A44",
    "Raindrop": "#4C9A2A",
    "ViTST-Swin": "#D149A0",
    "ViTST-ViT": "#6C757D",
}


def plot_masked_reconstruction_variants_overlay(
    variant_to_df_pid,
    *,
    var_names=None,
    plot_only_vars=None,
    time_col="times",
    var_col="varis",
    value_col="values",
    mask_col="mask",
    time_unit="min",
    figsize_per_plot=(4.8, 3.4),
    sharey=False,
    save_path=None,
    suptitle=None,
    fixed_var_ylims=None,
    variant_color_map=None,
    masked_color="#6BA06B",
):
    """
    Parameters
    ----------
    variant_to_df_pid : dict[str, pd.DataFrame]
        Example:
            {
                "Surprise": df_surprise_pid,
                "SurpVT": df_surpvt_pid,
                "SurpVTTG": df_surpvttg_pid,
            }

    Notes
    -----
    - Reconstruction line + unmasked points use the SAME color per variant.
    - Masked points use one shared color for all variants.
    - Legend order is exactly: variant / variant / ... / masked
    """
    if variant_color_map is None:
        variant_color_map = VARIANT_COLOR_MAP

    if not variant_to_df_pid:
        raise ValueError("variant_to_df_pid is empty.")

    # determine common vars across variants
    recon_all = {}
    common_vars = None

    for variant, df_pid in variant_to_df_pid.items():
        req = {"pid", time_col, var_col, value_col, mask_col}
        if not req.issubset(df_pid.columns):
            raise ValueError(f"{variant}: df must contain columns {req}")

        recon_dict = compute_reconstruction_dict(
            df_pid,
            time_col=time_col,
            var_col=var_col,
            value_col=value_col,
            mask_col=mask_col,
        )
        recon_all[variant] = recon_dict

        var_set = set(recon_dict.keys())
        if common_vars is None:
            common_vars = var_set
        else:
            common_vars = common_vars & var_set

    uniq_vars = sorted(common_vars)
    if plot_only_vars is not None:
        plot_set = set(plot_only_vars)
        uniq_vars = [v for v in uniq_vars if v in plot_set]

    nplots = len(uniq_vars)
    if nplots == 0:
        raise ValueError("No common variables to plot after filtering.")

    cols = int(ceil(sqrt(nplots)))
    rows_ = int(ceil(nplots / cols))

    fig, axes = plt.subplots(
        rows_, cols,
        figsize=(figsize_per_plot[0] * cols, figsize_per_plot[1] * rows_ + 0.9),
        sharey=sharey
    )
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.reshape(rows_, cols)

    variant_order = list(variant_to_df_pid.keys())

    for idx, var_id in enumerate(uniq_vars):
        r, c = divmod(idx, cols)
        ax = axes[r, c]

        masked_drawn = False

        for variant in variant_order:
            rd = recon_all[variant][var_id]
            color = variant_color_map.get(variant, None)

            # masked: draw only once per subplot using first variant
            if (not masked_drawn) and len(rd["t_mk"]) > 0:
                ax.scatter(
                    rd["t_mk"], rd["y_mk"],
                    marker='.',
                    alpha=0.85,
                    color=masked_color,
                    zorder=1,
                )
                masked_drawn = True

            # unmasked points
            if len(rd["t_un"]) > 0:
                ax.scatter(
                    rd["t_un"], rd["y_un"],
                    marker='o',
                    s=18,
                    alpha=0.95,
                    color=color,
                    zorder=2,
                )

            # reconstruction
            # ===== extend reconstruction to full time range =====
            if len(rd["t_un"]) >= 1:

                t_grid = np.linspace(0.0, 2880.0, 500)

                if len(rd["t_un"]) == 1:
                    y_hat_grid = np.full_like(
                        t_grid,
                        fill_value=rd["y_un"][0],
                        dtype=float
                    )
                else:
                    t_k = rd["t_un"]
                    y_k = rd["y_un"]

                    if len(np.unique(t_k)) != len(t_k):
                        tmp = (
                            pd.DataFrame({"t": t_k, "y": y_k})
                            .groupby("t")["y"].mean()
                            .reset_index()
                        )
                        t_k = tmp["t"].to_numpy()
                        y_k = tmp["y"].to_numpy()

                    order = np.argsort(t_k)
                    t_k = t_k[order]
                    y_k = y_k[order]

                    y_hat_grid = np.interp(t_grid, t_k, y_k)

                ax.plot(
                    t_grid,
                    y_hat_grid,
                    linewidth=1.4,
                    color=color,
                    alpha=0.95,
                    zorder=3,
                )

        ax.set_title(pretty_var_label_from_id(var_id, var_names=var_names), fontsize=20)
        ax.set_xlabel(f"Time ({time_unit})", fontsize=14)
        ax.set_ylabel("Value", fontsize=14)
        ax.set_xlim(0.0, 2880.0)
        ax.tick_params(axis='both', labelsize=12)

        ylim = get_fixed_ylim_for_var(var_id, var_names=var_names, fixed_var_ylims=fixed_var_ylims)
        if ylim is not None:
            ax.set_ylim(*ylim)

        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

    for j in range(nplots, rows_ * cols):
        r, c = divmod(j, cols)
        axes[r, c].axis('off')

    # custom legend: variant / variant / variant / masked
    legend_handles = []
    for variant in variant_order:
        color = variant_color_map.get(variant, None)
        legend_handles.append(
            Line2D(
                [0], [0],
                color=color,
                marker='o',
                linewidth=1.8,
                markersize=5,
                label=variant,
            )
        )

    legend_handles.append(
        Line2D(
            [0], [0],
            color=masked_color,
            marker='.',
            linewidth=0,
            markersize=10,
            label='Masked',
        )
    )

    fig.legend(
        handles=legend_handles,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.01),
        ncol=min(len(legend_handles), 4),
        frameon=False,
        fontsize=14
    )

    if suptitle is not None:
        fig.suptitle(suptitle, y=0.98, fontsize=24)
        fig.tight_layout(rect=[0.02, 0.10, 0.98, 0.94])
    else:
        fig.tight_layout(rect=[0.02, 0.10, 0.98, 0.98])

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()
    plt.close(fig)
# ============================================================
# Defaults (argparse defaults) -> args dict
# ============================================================
def make_default_args(
    *,
    model_name: str = "surprise",
    source_domain: str = "mimic",
    target_domain: str = "eicu",
    data_root: str = "./data",
    output_directory: str = "./surprise_results",
    project_name: str = "CV_5fold",
    max_epochs_pretrain: int = 30,
    max_epochs_train: int = 30,
    batch_size: int = 8,
    num_workers: int = 0,
    lr: float = 1e-3,
    weight_decay: float = 1e-2,
    optimizer: str = "adamw",
    task_type: str = "multilabel",
    seed: int = 9871,
    do_pretrain: bool = True,
    pre_mask_p_train: float = 0.15,
    earlystop_pretrain_patience: int = 7,
    earlystop_train_patience: int = 7,
    precision: str = "16-mixed",
    accelerator: str = "auto",
    devices: str = "auto",
    wandb_entity: str = "jwseo118-korea-university",
    log_to_wandb: bool = True,
    model_cfg_json: str = "{}",
    num_splits: int = 5,
    fixed_target_test_split: int = 1,
):
    return Namespace(
        project_name=project_name,
        wandb_entity=wandb_entity,
        data_root=data_root,
        source_domain=source_domain,
        target_domain=target_domain,
        num_splits=num_splits,
        fixed_target_test_split=fixed_target_test_split,

        model_name=model_name,
        model_cfg_json=model_cfg_json,

        task_type=task_type,

        batch_size=batch_size,
        num_workers=num_workers,
        lr=lr,
        weight_decay=weight_decay,
        optimizer=optimizer,

        precision=precision,
        accelerator=accelerator,
        devices=devices,
        seed=seed,

        do_pretrain=str(do_pretrain),
        max_epochs_pretrain=max_epochs_pretrain,
        pre_mask_p_train=pre_mask_p_train,
        earlystop_pretrain_patience=earlystop_pretrain_patience,

        max_epochs_train=max_epochs_train,
        earlystop_train_patience=earlystop_train_patience,

        log_to_wandb=str(log_to_wandb),
        output_directory=output_directory,
    )


# ============================================================
# Helpers: ckpt, embeddings, pid lookup, check_padding
# ============================================================

def find_best_ckpt(
    *,
    output_directory: str,
    project_name: str,
    split_id: int,
    stage: str,                  # "pretrain" or "train"
    monitor: str,                # "pretrain_val_loss" or "val_AUPRC_macro"
    mode: str,                   # "min" or "max"
    model_name: str,
    source_domain: str,
):
    """
    Select best checkpoint using metric encoded in filename:
    example:
        surprise_vttg_eicu_1-train-epoch=18-val_AUPRC_macro=0.7123.ckpt
    """

    split_out = os.path.join(output_directory, f"{project_name}_split{split_id}")
    ckpt_dir = os.path.join(split_out, stage, "checkpoints")

    run_name = f"{model_name}_{source_domain}_{split_id}"
    prefix = run_name.replace("/", "_").replace("\\", "_").replace(" ", "_")

    paths = glob.glob(os.path.join(ckpt_dir, f"{prefix}-{stage}-*.ckpt"))

    if not paths:
        raise FileNotFoundError(f"No ckpts found in {ckpt_dir}")

    pattern = re.compile(rf"{re.escape(monitor)}=([0-9]+(?:\.[0-9]+)?)")
    scored = []
    for p in paths:
        fname = os.path.basename(p)
        m = pattern.search(fname)
        if m:
            score = float(m.group(1))
            scored.append((score, p))

    if not scored:
        raise RuntimeError(f"Could not parse metric '{monitor}' from checkpoint filenames.")

    if mode == "max":
        best = max(scored, key=lambda x: x[0])
    else:
        best = min(scored, key=lambda x: x[0])

    best_score, best_path = best
    print(f"[ckpt] selected {best_path} ({monitor}={best_score:.4f})")

    return best_path


@torch.no_grad()
def collect_embeddings(net, dataloader, device: str):
    embs_list, pids_list = [], []
    for batch in dataloader:
        times, varis, values, pad_mask, _pre_mask, _y, pid, static = batch
        out = net(
            times=times.to(device),
            varis=varis.to(device),
            values=values.to(device),
            statics=static.to(device),
            padding_mask=pad_mask.to(device),
            pretrain=False,
        )
        embs_list.append(out["embs"].detach().cpu())
        pids_list.append(pid.detach().cpu() if torch.is_tensor(pid) else torch.tensor(pid))
    return torch.cat(embs_list, 0), torch.cat(pids_list, 0)


def find_instance_by_pid(dataloader, pid_value: int):
    for batch in dataloader:
        times, varis, values, pad_mask, _pre_mask, _y, pid, _static = batch
        if torch.is_tensor(pid):
            m = (pid == pid_value)
            if m.any():
                idx = int(torch.nonzero(m, as_tuple=False)[0].item())
                return (times[idx:idx+1], varis[idx:idx+1], values[idx:idx+1], pad_mask[idx:idx+1])
        else:
            if pid_value in pid:
                idx = pid.index(pid_value)
                return (times[idx:idx+1], varis[idx:idx+1], values[idx:idx+1], pad_mask[idx:idx+1])
    return None


@torch.no_grad()
def run_check_padding(net, times1, varis1, values1, pad1, device: str):
    dbg = net.check_padding(
        times=times1.to(device),
        varis=varis1.to(device),
        values=values1.to(device),
        padding_mask=pad1.to(device),
    )
    return {k: (v.detach().cpu() if torch.is_tensor(v) else v) for k, v in dbg.items()}


# ============================================================
# PaCMAP plot
# ============================================================
def plot_pacmap_two_domains(
    embs_source: torch.Tensor,
    embs_target: torch.Tensor,
    *,
    title: str,
    save_path=None,
    n_neighbors: int = 10,
    MN_ratio: float = 0.5,
    FP_ratio: float = 2.0,
    random_state: int = 0,
    max_points_per_domain: int | None = 5000,
):
    Xs = embs_source.detach().cpu().numpy()
    Xt = embs_target.detach().cpu().numpy()
    rng = np.random.default_rng(random_state)

    if max_points_per_domain is not None:
        if len(Xs) > max_points_per_domain:
            Xs = Xs[rng.choice(len(Xs), size=max_points_per_domain, replace=False)]
        if len(Xt) > max_points_per_domain:
            Xt = Xt[rng.choice(len(Xt), size=max_points_per_domain, replace=False)]

    X = np.concatenate([Xs, Xt], axis=0)
    y = np.array([0]*len(Xs) + [1]*len(Xt), dtype=int)

    reducer = pacmap.PaCMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        MN_ratio=MN_ratio,
        FP_ratio=FP_ratio,
        random_state=random_state,
    )
    Z = reducer.fit_transform(X)

    plt.figure(figsize=(6, 5))
    plt.scatter(Z[y==0, 0], Z[y==0, 1], s=8, alpha=0.5, color="#2D2926", label="Source")
    plt.scatter(Z[y==1, 0], Z[y==1, 1], s=8, alpha=0.5, color="#ed6f63", label="Target")
    plt.title(title, fontsize=22)
    plt.legend(frameon=False, fontsize=14)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)

    plt.show()
    plt.close()
    return Z, y


# ============================================================
# NEW: build per-PID long DataFrame from tensors + mask
# ============================================================
def pid_tensors_to_long_df(
    *,
    pid: int,
    times1: torch.Tensor,              # [1, L]
    varis1: torch.Tensor,              # [1, L]
    values1: torch.Tensor,             # [1, L]
    pad1: torch.Tensor,                # [1, L] bool, True = pad
    redundant_mask1: torch.Tensor,     # [1, L] bool, True = masked/redundant
    inverse_time_fn=None,
    inverse_value_map=None,
) -> pd.DataFrame:
    """
    Convert one patient's tensor representation into a long-form DataFrame.

    Returns columns:
        pid, times, varis, values, mask

    Notes
    -----
    - padded positions are removed
    - mask=True means masked/redundant
    - if inverse_time_fn is given, times are mapped back to original scale
    - if inverse_value_map is given, values are inverse-transformed per variable
    """
    t = times1.detach().cpu().reshape(-1).numpy()
    f = varis1.detach().cpu().reshape(-1).numpy()
    v = values1.detach().cpu().reshape(-1).numpy()
    pad = pad1.detach().cpu().reshape(-1).numpy().astype(bool)
    m = redundant_mask1.detach().cpu().reshape(-1).numpy().astype(bool)

    keep = ~pad
    t = t[keep]
    f = f[keep]
    v = v[keep]
    m = m[keep]

    if inverse_time_fn is not None:
        t = inverse_time_fn(t)

    if inverse_value_map is not None:
        v_out = np.asarray(v, dtype=float).copy()
        for var_id, fn in inverse_value_map.items():
            sel = (f == var_id)
            if np.any(sel):
                v_out[sel] = fn(v_out[sel])
        v = v_out

    df = pd.DataFrame({
        "pid": pid,
        "times": t,
        "varis": f,
        "values": v,
        "mask": m,
    })
    return df

@torch.no_grad()
def collect_mask_counts_over_loader(
    net,
    dataloader,
    *,
    device: str,
    var_names: dict[int, str] | None = None,
):
    total_masked_count = 0
    total_nonpad_count = 0
    per_var_counts = {}

    for batch in dataloader:
        times, varis, values, pad_mask, _pre_mask, _y, pid, static = batch

        dbg = net.check_padding(
            times=times.to(device),
            varis=varis.to(device),
            values=values.to(device),
            padding_mask=pad_mask.to(device),
        )

        varis_cpu = varis.detach().cpu()
        pad_cpu = pad_mask.detach().cpu().bool()
        mask_cpu = dbg["mask"].detach().cpu().bool()

        valid = ~pad_cpu
        masked_valid = mask_cpu & valid

        total_masked_count += int(masked_valid.sum().item())
        total_nonpad_count += int(valid.sum().item())

        masked_varis = varis_cpu[masked_valid]
        if masked_varis.numel() > 0:
            uniq, counts = torch.unique(masked_varis, return_counts=True)
            for v, c in zip(uniq.tolist(), counts.tolist()):
                per_var_counts[int(v)] = per_var_counts.get(int(v), 0) + int(c)

    if per_var_counts:
        rows = []
        for v, c in sorted(per_var_counts.items(), key=lambda x: x[1], reverse=True):
            rows.append({
                "varis": v,
                "var_name": var_names.get(v, f"var {v}") if var_names else f"var {v}",
                "masked_count": c,
            })
        var_count_df = pd.DataFrame(rows)
    else:
        var_count_df = pd.DataFrame(columns=["varis", "var_name", "masked_count"])

    overall = {
        "total_masked_count": int(total_masked_count),
        "total_nonpad_count": int(total_nonpad_count),
        "mask_ratio": (total_masked_count / total_nonpad_count) if total_nonpad_count > 0 else np.nan,
    }

    return var_count_df, overall


def inverse_time_default(t_norm):
    """
    Inverse-transform normalized time to original minutes.

    Assumption:
        normalized time in [0, 1]
        original time in [0, 2880]
    """
    t_norm = np.asarray(t_norm, dtype=float)
    return t_norm * 2880.0


def make_inverse_value_map_from_stats(stats):
    """
    Build per-item inverse transform functions from normalization stats.

    Expected stats structure (based on normalize_single_samples usage):
        stats["value"]["per_item"] : dict[itemid] -> (mean, std)
        stats["value"]["global"]   : (global_mean, global_std)
        stats["value"]["categorical"] : iterable of categorical itemids (optional)

    Returns
    -------
    inverse_value_map : dict[itemid, callable]
        Each callable maps normalized values -> original values.
        For categorical variables, identity is used.
    """
    if "value" not in stats:
        raise KeyError("stats must contain stats['value'].")

    value_stats = stats["value"]

    if "per_item" not in value_stats:
        raise KeyError("stats['value'] must contain 'per_item'.")
    if "global" not in value_stats:
        raise KeyError("stats['value'] must contain 'global'.")

    per_item = value_stats["per_item"]
    global_mean, global_std = value_stats["global"]
    categorical_itemids = set(value_stats.get("categorical", []))

    if abs(global_std) < 1e-12:
        global_std = 1.0

    inverse_value_map = {}

    all_itemids = set(per_item.keys()) | categorical_itemids

    for itemid in all_itemids:
        if itemid in categorical_itemids:
            inverse_value_map[itemid] = lambda x: np.asarray(x)
        else:
            mean_i, std_i = per_item.get(itemid, (global_mean, global_std))
            if abs(std_i) < 1e-12:
                std_i = 1.0

            inverse_value_map[itemid] = (
                lambda x, mu=mean_i, sd=std_i: np.asarray(x, dtype=float) * sd + mu
            )

    return inverse_value_map
# ============================================================
# Main pipeline (with PaCMAP + masked reconstruction plotting)
# ============================================================
def run_split1_pipeline(
    *,
    model_name: str,
    source_domain: str,
    source_split_id: int = 1,
    target_domain: str | None = None,
    fixed_target_test_split: int = 1,   # kept for compatibility
    pid_source: int | None = None,
    pid_target: int | None = None,
    device: str | None = None,
    data_root: str = "./data",
    output_directory: str = "./surprise_results",
    project_name: str = "CV_5fold",
    model_cfg_json: str = "{}",

    # plotting flags
    do_pacmap: bool = True,
    do_masked_recon_plot: bool = True,
    use_seaborn: bool = False,

    # selected variables by NAME, not id
    plot_only_var_names: list[str] | None = None,
    var_names: dict[int, str] | None = None,

    # if True, automatically build inverse map from returned stats
    auto_inverse_values_from_stats: bool = True,

    # optional manual overrides
    inverse_time_fn=None,
    inverse_value_map=None,
    

    # saving flags
    save_mask_count_csv: bool = True,
):
    """
    Pipeline for:
      1) loading best checkpoint for a source split
      2) collecting source/target test embeddings
      3) optional PaCMAP visualization
      4) optional masked reconstruction plot for one source pid and one target pid
      5) if model_name != "strats", summarize test-set mask counts over full source/target loaders
      6) keep normalization stats for inverse-transform visualization

    Returns
    -------
    dict with cfg, ckpt path, embeddings, pacmap outputs,
    optional mask summaries, optional reconstruction summaries,
    and visualization metadata including stats/raw-normalized samples.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if target_domain is None:
        target_domain = "eicu" if source_domain == "mimic" else "mimic"

    # ------------------------------------------------------------
    # output dirs
    # ------------------------------------------------------------
    split_out = os.path.join(output_directory, f"{project_name}_split{source_split_id}")
    fig_dir = os.path.join(split_out, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    # ------------------------------------------------------------
    # outcome columns
    # ------------------------------------------------------------
    ocs_path = os.path.join(data_root, "split1", f"{source_domain}_outcomes_train_1.feather")
    ocs_df = pd.read_feather(ocs_path)
    ocs = ocs_df.columns.tolist()
    if "pid" in ocs:
        ocs.remove("pid")

    # ------------------------------------------------------------
    # cfg
    # ------------------------------------------------------------
    args = make_default_args(
        model_name=model_name,
        source_domain=source_domain,
        target_domain=target_domain,
        data_root=data_root,
        output_directory=output_directory,
        project_name=project_name,
        model_cfg_json=model_cfg_json,
        fixed_target_test_split=fixed_target_test_split,
    )
    cfg = build_feather_cfg_from_args(args, ocs)

    if not hasattr(cfg, "model_cfg") or cfg.model_cfg is None:
        cfg.model_cfg = json.loads(model_cfg_json)

    # ------------------------------------------------------------
    # checkpoint
    # ------------------------------------------------------------
    best_ckpt = find_best_ckpt(
        output_directory=cfg.output_directory,
        project_name=cfg.project_name,
        split_id=source_split_id,
        stage="train",
        monitor="val_AUPRC_macro",
        mode="max",
        model_name=model_name,
        source_domain=source_domain,
    )
    print(f"[ckpt] {best_ckpt}")

    # ------------------------------------------------------------
    # loaders + stats for visualization
    # ------------------------------------------------------------
    vis_bundle = build_loaders_for_vis(cfg, split_id=int(source_split_id), return_raw=True)

    num_features = vis_bundle["num_features"]
    pre_ld_tr = vis_bundle["pretrain_train_loader"]
    pre_ld_va = vis_bundle["pretrain_val_loader"]
    ld_tr = vis_bundle["train_loader"]
    ld_va = vis_bundle["val_loader"]
    ld_te = vis_bundle["source_test_loader"]
    target_loader = vis_bundle["target_test_loader"]
    stats = vis_bundle["stats"]

    # ------------------------------------------------------------
    # selected variables by NAME -> itemid
    # ------------------------------------------------------------
    plot_only_vars = None
    if plot_only_var_names is not None:
        if var_names is None:
            raise ValueError("plot_only_var_names requires var_names.")
        inv_var_names = invert_var_names(var_names)
        missing = [n for n in plot_only_var_names if n not in inv_var_names]
        if missing:
            raise ValueError(f"Unknown variable names in plot_only_var_names: {missing}")
        plot_only_vars = [inv_var_names[n] for n in plot_only_var_names]

    # ------------------------------------------------------------
    # inverse transforms for plotting
    # ------------------------------------------------------------
    if inverse_time_fn is None:
        inverse_time_fn = inverse_time_default

    if inverse_value_map is None and auto_inverse_values_from_stats:
        inverse_value_map = make_inverse_value_map_from_stats(stats)

    # ------------------------------------------------------------
    # model restore
    # ------------------------------------------------------------
    n_output = getattr(cfg, "n_output", 1)
    core_model = build_model(
        cfg.model_name,
        cfg.model_cfg,
        num_features=num_features,
        n_output=n_output,
    )

    lit = STRaTSLit(
        model=core_model,
        task_type=cfg.task_type,
        outcome_cols=cfg.outcome_cols,
        target_indices=cfg.target_indices,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        optimizer=cfg.optimizer,
    )

    ckpt = torch.load(best_ckpt, map_location="cpu")
    lit.load_state_dict(ckpt["state_dict"], strict=False)
    lit.eval().to(device)
    net = lit.model.eval()

    # ------------------------------------------------------------
    # embeddings
    # ------------------------------------------------------------
    # src_embs, src_pids = collect_embeddings(net, ld_te, device=device)
    # tgt_embs, tgt_pids = collect_embeddings(net, target_loader, device=device)

    # print("[SOURCE test] embs:", tuple(src_embs.shape))
    # print("[TARGET test] embs:", tuple(tgt_embs.shape))

    # ------------------------------------------------------------
    # default PID selection for visualization
    # ------------------------------------------------------------
    # if pid_source is None:
    #     pid_source = int(src_pids[0].item())
    # if pid_target is None:
    #     pid_target = int(tgt_pids[0].item())

    # ------------------------------------------------------------
    # whole test-set mask count summary
    # only for non-STRaTS models
    # ------------------------------------------------------------
    src_mask_count_df = None
    src_mask_overall = None
    tgt_mask_count_df = None
    tgt_mask_overall = None

    # if model_name != "strats":
    #     print("\n[Mask count summary] SOURCE test set")
    #     src_mask_count_df, src_mask_overall = collect_mask_counts_over_loader(
    #         net,
    #         ld_te,
    #         device=device,
    #         var_names=var_names,
    #     )
    #     print(src_mask_count_df.head(20))
    #     print(src_mask_overall)

    #     print("\n[Mask count summary] TARGET test set")
    #     tgt_mask_count_df, tgt_mask_overall = collect_mask_counts_over_loader(
    #         net,
    #         target_loader,
    #         device=device,
    #         var_names=var_names,
    #     )
    #     print(tgt_mask_count_df.head(20))
    #     print(tgt_mask_overall)

    #     if save_mask_count_csv:
    #         src_mask_count_df.to_csv(
    #             os.path.join(fig_dir, f"mask_count_source_{model_name}_{source_domain}.csv"),
    #             index=False,
    #         )
    #         pd.DataFrame([src_mask_overall]).to_csv(
    #             os.path.join(fig_dir, f"mask_count_source_total_{model_name}_{source_domain}.csv"),
    #             index=False,
    #         )
    #         tgt_mask_count_df.to_csv(
    #             os.path.join(fig_dir, f"mask_count_target_{model_name}_{target_domain}.csv"),
    #             index=False,
    #         )
    #         pd.DataFrame([tgt_mask_overall]).to_csv(
    #             os.path.join(fig_dir, f"mask_count_target_total_{model_name}_{target_domain}.csv"),
    #             index=False,
    #         )

    # ------------------------------------------------------------
    # PaCMAP
    # ------------------------------------------------------------
    pacmap_Z, pacmap_y = None, None
    pacmap_path = None

    # initialize as None so return dict doesn't break
    src_embs, src_pids = None, None
    tgt_embs, tgt_pids = None, None

    if do_pacmap:
        # 🔥 embeddings ONLY computed here
        src_embs, src_pids = collect_embeddings(net, ld_te, device=device)
        tgt_embs, tgt_pids = collect_embeddings(net, target_loader, device=device)

        print("[SOURCE test] embs:", tuple(src_embs.shape))
        print("[TARGET test] embs:", tuple(tgt_embs.shape))

        plot_name = format_model_name(model_name)
        plot_source = format_source_name(source_domain)

        title = f"PaCMAP: {plot_name} | Source={plot_source}"

        pacmap_path = os.path.join(
            fig_dir,
            f"pacmap_{model_name}_{source_domain}_to_{target_domain}.png",
        )

        pacmap_Z, pacmap_y = plot_pacmap_two_domains(
            src_embs,
            tgt_embs,
            title=title,
            random_state=int(getattr(cfg, "seed", 0)),
            save_path=pacmap_path,
        )

    # ------------------------------------------------------------
    # reconstruction visualization
    # ------------------------------------------------------------
    src_dbg = None
    tgt_dbg = None
    src_var_mse_df = None
    src_overall = None
    src_pid_var_detail = None
    tgt_var_mse_df = None
    tgt_overall = None
    tgt_pid_var_detail = None

    src_recon_path = None
    tgt_recon_path = None

    if do_masked_recon_plot:
        if model_name == "strats":
            print("\n[masked reconstruction plot skipped] model_name='strats' has no learned masking.")
        else:
            src_recon_path = os.path.join(
                fig_dir,
                f"reconstruction_source_{model_name}_{source_domain}_pid{pid_source}.png",
            )
            tgt_recon_path = os.path.join(
                fig_dir,
                f"reconstruction_target_{model_name}_{target_domain}_pid{pid_target}.png",
            )

            inst_src = find_instance_by_pid(ld_te, pid_source)
            if inst_src is None:
                raise ValueError(f"[SOURCE] pid={pid_source} not found in source test loader.")
            src_dbg = run_check_padding(net, *inst_src, device=device)

            inst_tgt = find_instance_by_pid(target_loader, pid_target)
            if inst_tgt is None:
                raise ValueError(f"[TARGET] pid={pid_target} not found in target test loader.")
            tgt_dbg = run_check_padding(net, *inst_tgt, device=device)

            print("[SOURCE] pid:", pid_source, "redundant:", int(src_dbg["mask"].sum().item()))
            print("[TARGET] pid:", pid_target, "redundant:", int(tgt_dbg["mask"].sum().item()))

            src_df_pid = pid_tensors_to_long_df(
                pid=pid_source,
                times1=src_dbg["times"],
                varis1=src_dbg["varis"],
                values1=src_dbg["values"],
                pad1=inst_src[3],
                redundant_mask1=src_dbg["mask"],
                inverse_time_fn=inverse_time_fn,
                inverse_value_map=inverse_value_map,
            )

            tgt_df_pid = pid_tensors_to_long_df(
                pid=pid_target,
                times1=tgt_dbg["times"],
                varis1=tgt_dbg["varis"],
                values1=tgt_dbg["values"],
                pad1=inst_tgt[3],
                redundant_mask1=tgt_dbg["mask"],
                inverse_time_fn=inverse_time_fn,
                inverse_value_map=inverse_value_map,
            )

            model_name_formatted = format_model_name(model_name)
            source_pretty = format_source_name(source_domain)
            target_pretty = format_source_name(target_domain)

            src_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={source_pretty}"
            tgt_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={target_pretty}"

            print("\n[SOURCE] masked reconstruction plot")
            src_var_mse_df, src_overall, src_pid_var_detail = masked_reconstruction_mse(
                src_df_pid,
                var_names=var_names,
                pid_filter=pid_source,
                pid_target=pid_source,
                plot_only_vars=plot_only_vars,
                time_col="times",
                var_col="varis",
                value_col="values",
                mask_col="mask",
                time_unit="min",
                save_path=src_recon_path,
                suptitle=src_title,
                fixed_var_ylims=FIXED_VAR_YLIMS,
            )

            print("\n[TARGET] masked reconstruction plot")
            tgt_var_mse_df, tgt_overall, tgt_pid_var_detail = masked_reconstruction_mse(
                tgt_df_pid,
                var_names=var_names,
                pid_filter=pid_target,
                pid_target=pid_target,
                plot_only_vars=plot_only_vars,
                time_col="times",
                var_col="varis",
                value_col="values",
                mask_col="mask",
                time_unit="min",
                save_path=tgt_recon_path,
                suptitle=tgt_title,
                fixed_var_ylims=FIXED_VAR_YLIMS,
            )

    # ------------------------------------------------------------
    # return everything useful
    # ------------------------------------------------------------
    return {
        "cfg": cfg,
        "best_ckpt": best_ckpt,

        "num_features": num_features,

        "pre_ld_tr": pre_ld_tr,
        "pre_ld_va": pre_ld_va,
        "ld_tr": ld_tr,
        "ld_va": ld_va,
        "ld_te": ld_te,
        "target_loader": target_loader,

        "stats": stats,
        "vis_bundle": vis_bundle,
        "inverse_time_fn": inverse_time_fn,
        "inverse_value_map": inverse_value_map,

        "src_embs": src_embs,
        "src_pids": src_pids,
        "tgt_embs": tgt_embs,
        "tgt_pids": tgt_pids,

        "src_pid": pid_source,
        "tgt_pid": pid_target,

        "src_check_padding": src_dbg,
        "tgt_check_padding": tgt_dbg,

        "pacmap_Z": pacmap_Z,
        "pacmap_domain_y": pacmap_y,
        "pacmap_path": pacmap_path,

        "src_mask_count_df": src_mask_count_df,
        "src_mask_overall": src_mask_overall,
        "tgt_mask_count_df": tgt_mask_count_df,
        "tgt_mask_overall": tgt_mask_overall,

        "src_var_mse_df": src_var_mse_df,
        "src_overall_mse": src_overall,
        "src_pid_var_detail": src_pid_var_detail,

        "tgt_var_mse_df": tgt_var_mse_df,
        "tgt_overall_mse": tgt_overall,
        "tgt_pid_var_detail": tgt_pid_var_detail,

        "src_df_pid": src_df_pid if do_masked_recon_plot and model_name != "strats" else None,
        "tgt_df_pid": tgt_df_pid if do_masked_recon_plot and model_name != "strats" else None,

        "figure_dir": fig_dir,
        "src_recon_path": src_recon_path,
        "tgt_recon_path": tgt_recon_path,
    }

def make_source_target_variant_overlays(
    variant_results,
    *,
    var_names,
    plot_only_var_names=None,
    save_dir=None,
    source_overlay_filename="overlay_variants_source.png",
    target_overlay_filename="overlay_variants_target.png",
    fixed_var_ylims=FIXED_VAR_YLIMS,
):
    """
    Build TWO overlay plots from multiple run_split1_pipeline results:
      1) source-data overlay across variants
      2) target-data overlay across variants

    Parameters
    ----------
    variant_results : dict[str, dict]
        Example:
        {
            "Surprise": res_surprise,
            "SurpVT": res_surpvt,
            "SurpVTTG": res_surpvttg,
        }

        Each value must be the returned dict from run_split1_pipeline(...),
        and must contain:
            - src_df_pid
            - tgt_df_pid
            - figure_dir
            - cfg or source/target metadata

    var_names : dict[int, str]
        Itemid -> variable name map.

    plot_only_var_names : list[str] | None
        Example: ["heart_rate", "map", "glucose", "lactate"]

    save_dir : str | None
        If None, uses figure_dir from the first result.

    Returns
    -------
    out : dict
        {
            "source_variant_to_df_pid": ...,
            "target_variant_to_df_pid": ...,
            "source_overlay_path": ...,
            "target_overlay_path": ...,
        }
    """
    if not variant_results:
        raise ValueError("variant_results is empty.")

    variant_names = list(variant_results.keys())
    first_res = variant_results[variant_names[0]]

    if save_dir is None:
        save_dir = first_res["figure_dir"]
    os.makedirs(save_dir, exist_ok=True)

    # variable names -> ids
    plot_only_vars = None
    if plot_only_var_names is not None:
        inv_var_names = invert_var_names(var_names)
        missing = [n for n in plot_only_var_names if n not in inv_var_names]
        if missing:
            raise ValueError(f"Unknown variable names in plot_only_var_names: {missing}")
        plot_only_vars = [inv_var_names[n] for n in plot_only_var_names]

    # collect per-variant source/target dfs
    source_variant_to_df_pid = {}
    target_variant_to_df_pid = {}

    for variant, res in variant_results.items():
        src_df_pid = res.get("src_df_pid", None)
        tgt_df_pid = res.get("tgt_df_pid", None)

        if src_df_pid is None:
            raise ValueError(f"{variant}: src_df_pid is missing. "
                             f"Modify run_split1_pipeline to return src_df_pid.")
        if tgt_df_pid is None:
            raise ValueError(f"{variant}: tgt_df_pid is missing. "
                             f"Modify run_split1_pipeline to return tgt_df_pid.")

        source_variant_to_df_pid[variant] = src_df_pid
        target_variant_to_df_pid[variant] = tgt_df_pid

    # dataset names from first result
    cfg = first_res.get("cfg", None)
    if cfg is not None:
        source_pretty = format_source_name(cfg.source_domain)
        target_pretty = format_source_name(cfg.target_domain)
    else:
        source_pretty = "Source"
        target_pretty = "Target"

    source_overlay_path = os.path.join(save_dir, source_overlay_filename)
    target_overlay_path = os.path.join(save_dir, target_overlay_filename)

    # ------------------------------------------------------------
    # source overlay across variants
    # ------------------------------------------------------------
    plot_masked_reconstruction_variants_overlay(
        source_variant_to_df_pid,
        var_names=var_names,
        plot_only_vars=plot_only_vars,
        suptitle=f"Masks from Variants | Data={source_pretty}",
        save_path=source_overlay_path,
        fixed_var_ylims=fixed_var_ylims,
    )

    # ------------------------------------------------------------
    # target overlay across variants
    # ------------------------------------------------------------
    plot_masked_reconstruction_variants_overlay(
        target_variant_to_df_pid,
        var_names=var_names,
        plot_only_vars=plot_only_vars,
        suptitle=f"Masks from Variants | Data={target_pretty}",
        save_path=target_overlay_path,
        fixed_var_ylims=fixed_var_ylims,
    )

    return {
        "source_variant_to_df_pid": source_variant_to_df_pid,
        "target_variant_to_df_pid": target_variant_to_df_pid,
        "source_overlay_path": source_overlay_path,
        "target_overlay_path": target_overlay_path,
    }

# ============================================================
# Example run
# ============================================================

var_names = {
    0: 'gcs',
    1: 'heart_rate',
    2: 'map',
    3: 'resp_rate',
    4: 'temperature',
    5: 'weight',
    6: 'albumin',
    7: 'bilirubin',
    8: 'creatinine',
    9: 'fio2',
    10: 'glucose',
    11: 'hematocrit',
    12: 'lactate',
    13: 'pao2',
    14: 'ph',
    15: 'platelets',
    16: 'sodium',
    17: 'urine',
    18: 'wbc',
    19: 'a10',
    20: 'a_drug',
    21: 'a_supplements',
    22: 'b',
    23: 'c01',
    24: 'c01_etc',
    25: 'c_else',
    26: 'h',
    27: 'l',
    28: 'm',
    29: 'n',
    30: 'r',
    31: 'v',
    32: 'antibiotics',
    33: 'fluid',
    34: 'ventilator',
}

selected_vars = [
    "heart_rate",
    "map",
    "resp_rate",
    "temperature",
    "lactate",
    "glucose",
]

### Run

In [ ]:
# ============================================================
# 1) valid pid 추출
# ============================================================
def get_valid_pids_with_required_vars(
    feather_path: str,
    required_itemids=(1, 2, 3, 4, 10),
):
    df = pd.read_feather(feather_path)

    counts = (
        df.groupby(["pid", "itemid"])
          .size()
          .unstack(fill_value=0)
    )

    required_itemids = list(required_itemids)
    for c in required_itemids:
        if c not in counts.columns:
            counts[c] = 0

    valid_pids = (
        counts.loc[(counts[required_itemids] > 0).all(axis=1)]
              .index
              .tolist()
    )
    return valid_pids


# ============================================================
# 2) 모델 1개를 한 번만 준비
# ============================================================
def prepare_variant_context(
    *,
    model_name: str,
    source_domain: str = "mimic",
    source_split_id: int = 1,
    target_domain: str | None = None,
    fixed_target_test_split: int = 1,
    device: str | None = None,
    data_root: str = "./data",
    output_directory: str = "./surprise_results",
    project_name: str = "CV_5fold",
    model_cfg_json: str = "{}",
    var_names: dict[int, str] | None = None,
    plot_only_var_names: list[str] | None = None,
    auto_inverse_values_from_stats: bool = True,
    inverse_time_fn=None,
    inverse_value_map=None,
    compute_loader_mask_summary: bool = True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if target_domain is None:
        target_domain = "eicu" if source_domain == "mimic" else "mimic"

    split_out = os.path.join(output_directory, f"{project_name}_split{source_split_id}")
    fig_dir = os.path.join(split_out, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    # outcome cols
    ocs_path = os.path.join(data_root, "split1", f"{source_domain}_outcomes_train_1.feather")
    ocs_df = pd.read_feather(ocs_path)
    ocs = ocs_df.columns.tolist()
    if "pid" in ocs:
        ocs.remove("pid")

    args = make_default_args(
        model_name=model_name,
        source_domain=source_domain,
        target_domain=target_domain,
        data_root=data_root,
        output_directory=output_directory,
        project_name=project_name,
        model_cfg_json=model_cfg_json,
        fixed_target_test_split=fixed_target_test_split,
    )
    cfg = build_feather_cfg_from_args(args, ocs)
    if not hasattr(cfg, "model_cfg") or cfg.model_cfg is None:
        cfg.model_cfg = json.loads(model_cfg_json)

    best_ckpt = find_best_ckpt(
        output_directory=cfg.output_directory,
        project_name=cfg.project_name,
        split_id=source_split_id,
        stage="train",
        monitor="val_AUPRC_macro",
        mode="max",
        model_name=model_name,
        source_domain=source_domain,
    )

    vis_bundle = build_loaders_for_vis(cfg, split_id=int(source_split_id), return_raw=True)

    num_features = vis_bundle["num_features"]
    ld_te = vis_bundle["source_test_loader"]
    target_loader = vis_bundle["target_test_loader"]
    stats = vis_bundle["stats"]

    if inverse_time_fn is None:
        inverse_time_fn = inverse_time_default
    if inverse_value_map is None and auto_inverse_values_from_stats:
        inverse_value_map = make_inverse_value_map_from_stats(stats)

    plot_only_vars = None
    if plot_only_var_names is not None:
        if var_names is None:
            raise ValueError("plot_only_var_names requires var_names.")
        inv_var_names = invert_var_names(var_names)
        missing = [n for n in plot_only_var_names if n not in inv_var_names]
        if missing:
            raise ValueError(f"Unknown variable names in plot_only_var_names: {missing}")
        plot_only_vars = [inv_var_names[n] for n in plot_only_var_names]

    # restore model once
    n_output = getattr(cfg, "n_output", 1)
    core_model = build_model(
        cfg.model_name,
        cfg.model_cfg,
        num_features=num_features,
        n_output=n_output,
    )

    lit = STRaTSLit(
        model=core_model,
        task_type=cfg.task_type,
        outcome_cols=cfg.outcome_cols,
        target_indices=cfg.target_indices,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        optimizer=cfg.optimizer,
    )

    ckpt = torch.load(best_ckpt, map_location="cpu")
    lit.load_state_dict(ckpt["state_dict"], strict=False)
    lit.eval().to(device)
    net = lit.model.eval()

    src_mask_count_df = None
    src_mask_overall = None
    tgt_mask_count_df = None
    tgt_mask_overall = None

    if compute_loader_mask_summary and model_name != "strats":
        src_mask_count_df, src_mask_overall = collect_mask_counts_over_loader(
            net, ld_te, device=device, var_names=var_names
        )
        tgt_mask_count_df, tgt_mask_overall = collect_mask_counts_over_loader(
            net, target_loader, device=device, var_names=var_names
        )

    return {
        "model_name": model_name,
        "cfg": cfg,
        "device": device,
        "best_ckpt": best_ckpt,
        "fig_dir": fig_dir,
        "net": net,
        "ld_te": ld_te,
        "target_loader": target_loader,
        "stats": stats,
        "inverse_time_fn": inverse_time_fn,
        "inverse_value_map": inverse_value_map,
        "plot_only_vars": plot_only_vars,
        "var_names": var_names,
        "source_domain": source_domain,
        "target_domain": target_domain,
        "src_mask_count_df": src_mask_count_df,
        "src_mask_overall": src_mask_overall,
        "tgt_mask_count_df": tgt_mask_count_df,
        "tgt_mask_overall": tgt_mask_overall,
    }


# ============================================================
# 3) context에서 pid pair 하나만 처리
# ============================================================
def run_pair_from_context(
    ctx,
    *,
    pid_source: int,
    pid_target: int,
    save_plots: bool = True,
):
    model_name = ctx["model_name"]
    net = ctx["net"]
    ld_te = ctx["ld_te"]
    target_loader = ctx["target_loader"]
    device = ctx["device"]
    inverse_time_fn = ctx["inverse_time_fn"]
    inverse_value_map = ctx["inverse_value_map"]
    plot_only_vars = ctx["plot_only_vars"]
    var_names = ctx["var_names"]
    source_domain = ctx["source_domain"]
    target_domain = ctx["target_domain"]
    fig_dir = ctx["fig_dir"]

    if model_name == "strats":
        raise ValueError("strats has no learned masking for this visualization.")

    inst_src = find_instance_by_pid(ld_te, pid_source)
    if inst_src is None:
        raise ValueError(f"[SOURCE] pid={pid_source} not found in source test loader.")

    inst_tgt = find_instance_by_pid(target_loader, pid_target)
    if inst_tgt is None:
        raise ValueError(f"[TARGET] pid={pid_target} not found in target test loader.")

    src_dbg = run_check_padding(net, *inst_src, device=device)
    tgt_dbg = run_check_padding(net, *inst_tgt, device=device)

    src_df_pid = pid_tensors_to_long_df(
        pid=pid_source,
        times1=src_dbg["times"],
        varis1=src_dbg["varis"],
        values1=src_dbg["values"],
        pad1=inst_src[3],
        redundant_mask1=src_dbg["mask"],
        inverse_time_fn=inverse_time_fn,
        inverse_value_map=inverse_value_map,
    )

    tgt_df_pid = pid_tensors_to_long_df(
        pid=pid_target,
        times1=tgt_dbg["times"],
        varis1=tgt_dbg["varis"],
        values1=tgt_dbg["values"],
        pad1=inst_tgt[3],
        redundant_mask1=tgt_dbg["mask"],
        inverse_time_fn=inverse_time_fn,
        inverse_value_map=inverse_value_map,
    )

    model_name_formatted = format_model_name(model_name)
    source_pretty = format_source_name(source_domain)
    target_pretty = format_source_name(target_domain)

    src_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={source_pretty}"
    tgt_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={target_pretty}"

    src_recon_path = None
    tgt_recon_path = None
    if save_plots:
        src_recon_path = os.path.join(
            fig_dir,
            f"reconstruction_source_{model_name}_{source_domain}_pid{pid_source}.png",
        )
        tgt_recon_path = os.path.join(
            fig_dir,
            f"reconstruction_target_{model_name}_{target_domain}_pid{pid_target}.png",
        )

    src_var_mse_df, src_overall, src_pid_var_detail = masked_reconstruction_mse(
        src_df_pid,
        var_names=var_names,
        pid_filter=pid_source,
        pid_target=pid_source,
        plot_only_vars=plot_only_vars,
        time_col="times",
        var_col="varis",
        value_col="values",
        mask_col="mask",
        time_unit="min",
        save_path=src_recon_path,
        suptitle=src_title,
        fixed_var_ylims=FIXED_VAR_YLIMS,
    )

    tgt_var_mse_df, tgt_overall, tgt_pid_var_detail = masked_reconstruction_mse(
        tgt_df_pid,
        var_names=var_names,
        pid_filter=pid_target,
        pid_target=pid_target,
        plot_only_vars=plot_only_vars,
        time_col="times",
        var_col="varis",
        value_col="values",
        mask_col="mask",
        time_unit="min",
        save_path=tgt_recon_path,
        suptitle=tgt_title,
        fixed_var_ylims=FIXED_VAR_YLIMS,
    )

    return {
        "model_name": model_name,
        "pid_source": pid_source,
        "pid_target": pid_target,
        "src_df_pid": src_df_pid,
        "tgt_df_pid": tgt_df_pid,
        "src_check_padding": src_dbg,
        "tgt_check_padding": tgt_dbg,
        "src_var_mse_df": src_var_mse_df,
        "src_overall_mse": src_overall,
        "src_pid_var_detail": src_pid_var_detail,
        "tgt_var_mse_df": tgt_var_mse_df,
        "tgt_overall_mse": tgt_overall,
        "tgt_pid_var_detail": tgt_pid_var_detail,
        "src_mask_overall": ctx["src_mask_overall"],
        "tgt_mask_overall": ctx["tgt_mask_overall"],
        "src_recon_path": src_recon_path,
        "tgt_recon_path": tgt_recon_path,
        "figure_dir": ctx["fig_dir"],
        "cfg": ctx["cfg"],
    }


# ============================================================
# 4) 여러 pid pair를 효율적으로 순회
# ============================================================
def run_all_pairs_efficient(
    *,
    source_domain="mimic",
    target_domain="eicu",
    source_split_id=1,
    data_root="./data",
    output_directory="./surprise_results",
    project_name="CV_5fold",
    model_cfg_json="{}",
    var_names=None,
    selected_vars=None,
    model_names=("surprise", "surprise_vt", "surprise_vttg"),
    required_itemids=(1, 2, 3, 4, 10),
    save_overlays=True,
):
    # valid pids once
    pid_m = get_valid_pids_with_required_vars(
        os.path.join(data_root, "split1", "mimic_data_test_1.feather"),
        required_itemids=required_itemids,
    )
    pid_e = get_valid_pids_with_required_vars(
        os.path.join(data_root, "split1", "eicu_data_test_1.feather"),
        required_itemids=required_itemids,
    )

    n_pairs = min(len(pid_m), len(pid_e))
    print(f"valid MIMIC pids: {len(pid_m)}")
    print(f"valid eICU pids: {len(pid_e)}")
    print(f"paired runs: {n_pairs}")

    # prepare each model once
    contexts = {}
    for model_name in model_names:
        print(f"\nPreparing context: {model_name}")
        contexts[model_name] = prepare_variant_context(
            model_name=model_name,
            source_domain=source_domain,
            source_split_id=source_split_id,
            target_domain=target_domain,
            data_root=data_root,
            output_directory=output_directory,
            project_name=project_name,
            model_cfg_json=model_cfg_json,
            var_names=var_names,
            plot_only_var_names=selected_vars,
            compute_loader_mask_summary=True,
        )

        print(f"[{model_name}] source loader mask summary:", contexts[model_name]["src_mask_overall"])
        print(f"[{model_name}] target loader mask summary:", contexts[model_name]["tgt_mask_overall"])

    all_results = []

    pretty_name_map = {
        "surprise": "Surprise",
        "surprise_vt": "SurpVT",
        "surprise_vttg": "SurpVTTG",
        "strats": "STraTS",
    }

    for i in range(n_pairs):
        pid_source = int(pid_m[i])
        pid_target = int(pid_e[i])

        print("\n" + "=" * 100)
        print(f"[{i+1}/{n_pairs}] pid_source={pid_source}, pid_target={pid_target}")

        pair_results = {}

        for model_name in model_names:
            res = run_pair_from_context(
                contexts[model_name],
                pid_source=pid_source,
                pid_target=pid_target,
                save_plots=True,
            )
            pair_results[pretty_name_map.get(model_name, model_name)] = res

            print(f"[{model_name}] source overall:", res["src_mask_overall"])
            print(f"[{model_name}] target overall:", res["tgt_mask_overall"])

        overlay_out = None
        if save_overlays:
            overlay_out = make_source_target_variant_overlays(
                pair_results,
                var_names=var_names,
                plot_only_var_names=selected_vars,
                source_overlay_filename=f"overlay_variants_source_{pid_source}.png",
                target_overlay_filename=f"overlay_variants_target_{pid_target}.png",
            )

        all_results.append({
            "i": i,
            "pid_source": pid_source,
            "pid_target": pid_target,
            "pair_results": pair_results,
            "overlay_out": overlay_out,
        })

    return {
        "pid_m": pid_m,
        "pid_e": pid_e,
        "contexts": contexts,
        "all_results": all_results,
    }

In [ ]:
selected_vars = ["heart_rate", "map", "resp_rate", "temperature", "glucose"]

out = run_all_pairs_efficient(
    source_domain="mimic",
    target_domain="eicu",
    source_split_id=1,
    data_root="./data",
    output_directory="./surprise_results",
    project_name="CV_5fold",
    var_names=var_names,
    selected_vars=selected_vars,
    model_names=("surprise", "surprise_vt", "surprise_vttg"),
    required_itemids=(1, 2, 3, 4, 10),
)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
# plt.rcParams['font.size'] = 22

#pid_source = 21206625
#pid_target = 2210186
# pid_target = 3239318
# pid_source = 20263113 surpvt가 다 가림
# pid_target = 579301
# pid_target = 613584 surpvt가 거의 못 따라

# 얘네는 surpvt만이 문제...
# pid_source = 20266641
# pid_target = 621660


# pid_source = 20271349
# pid_target = 637979

pid_source = 20311746
#pid_target = 692495 일단 보류
pid_target = 754429

# res1 = run_split1_pipeline(
#     model_name="strats",
#     source_domain="mimic",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=False,
#     do_masked_recon_plot=False,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res1['src_mask_overall'])
# print(res1['tgt_mask_overall'])

res2 = run_split1_pipeline(
    model_name="surprise",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res2['src_mask_overall'])
print(res2['tgt_mask_overall'])

res3 = run_split1_pipeline(
    model_name="surprise_vt",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res3['src_mask_overall'])
print(res3['tgt_mask_overall'])

res4 = run_split1_pipeline(
    model_name="surprise_vttg",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res4['src_mask_overall'])
print(res4['tgt_mask_overall'])

overlay_out = make_source_target_variant_overlays(
    {
        "Surprise": res2,
        "SurpVT": res3,
        "SurpVTTG": res4,
    },
    var_names=var_names,
    plot_only_var_names=selected_vars,
    source_overlay_filename=f"overlay_variants_source_{pid_source}.png",
    target_overlay_filename=f"overlay_variants_target_{pid_target}.png",
)

In [ ]:
overlay_out = make_source_target_variant_overlays(
    {
        "Surprise": res2,
        "SurpVT": res3,
        "SurpVTTG": res4,
    },
    var_names=var_names,
    plot_only_var_names=selected_vars,
)

In [ ]:
pid_source = 637979
pid_target = 20271349


res5 = run_split1_pipeline(
    model_name="strats",
    source_domain="eicu",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=True,
    do_masked_recon_plot=False,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res5['src_mask_overall'])
print(res5['tgt_mask_overall'])

res6 = run_split1_pipeline(
    model_name="surprise",
    source_domain="eicu",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=True,
    do_masked_recon_plot=False,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res6['src_mask_overall'])
print(res6['tgt_mask_overall'])

# res7 = run_split1_pipeline(
#     model_name="surprise_vt",
#     source_domain="eicu",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=True,
#     do_masked_recon_plot=True,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res7['src_mask_overall'])
# print(res7['tgt_mask_overall'])

# res8 = run_split1_pipeline(
#     model_name="surprise_vttg",
#     source_domain="eicu",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=True,
#     do_masked_recon_plot=True,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res8['src_mask_overall'])
# print(res8['tgt_mask_overall'])
